# Chapter 14 — Recommendation Systems

**Goals**

- Build a small synthetic user-item rating matrix.
- Implement content-based and collaborative-filtering predictors from scratch.
- Evaluate them on held-out ratings via RMSE / MAE.
- Compare against the global-mean baseline.

Pair with `docs/recommender_systems.md`. Real-world MovieLens is left as exercise E1.

## 1. Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=3, suppress=True)

## 2. A toy ratings matrix

10 users, 8 movies, ratings 1-5. We synthesise three latent user *taste vectors* and three movie *style vectors* so the data has real structure but is small enough to inspect by eye.

In [2]:
MOVIES = ['Action1', 'Action2', 'RomCom1', 'RomCom2', 'SciFi1', 'SciFi2', 'Drama1', 'Drama2']
GENRES = ['action', 'romance', 'scifi', 'drama']
MOVIE_GENRE = np.array([
    [1, 0, 0, 0],    # Action1
    [1, 0, 0, 0],    # Action2
    [0, 1, 0, 0],    # RomCom1
    [0, 1, 0, 0],    # RomCom2
    [0, 0, 1, 0],    # SciFi1
    [0, 0, 1, 0],    # SciFi2
    [0, 0, 0, 1],    # Drama1
    [0, 0, 0, 1],    # Drama2
], dtype=float)

USER_TASTE = np.array([
    [5, 1, 1, 2],    # u0 action lover
    [5, 1, 1, 2],
    [1, 5, 1, 2],    # u2 romcom lover
    [1, 5, 1, 2],
    [1, 1, 5, 2],    # u4 scifi lover
    [1, 1, 5, 2],
    [2, 2, 1, 5],    # u6 drama lover
    [2, 2, 1, 5],
    [4, 4, 4, 4],    # u8 generalist
    [3, 3, 3, 3],    # u9 generalist
], dtype=float)

N_USERS, N_MOVIES = USER_TASTE.shape[0], MOVIE_GENRE.shape[0]

# Generate full ratings 1-5 by dot-product of taste and genre then clip and add noise.
R_full = USER_TASTE @ MOVIE_GENRE.T / 5.0   # raw scores
R_full = np.clip(R_full + rng.normal(0, 0.3, size=R_full.shape), 1, 5)
R_full = np.round(R_full).astype(float)

R_full_df = pd.DataFrame(R_full, columns=MOVIES, index=[f'u{i}' for i in range(N_USERS)])
R_full_df

,Action1,Action2,RomCom1,RomCom2,SciFi1,SciFi2,Drama1,Drama2
u0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
u1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
u2,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
u3,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
u4,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
u5,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
u6,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
u7,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
u8,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
u9,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


## 3. Hold out 20% of ratings as test set

Real recommenders use a temporal split. For this toy we just randomly mask cells.

In [3]:
R = R_full.copy()
mask_pct = 0.2
test_mask = rng.random(R.shape) < mask_pct
R_train = R.copy()
R_train[test_mask] = np.nan

n_obs = np.sum(~np.isnan(R_train))
n_test = int(test_mask.sum())
print(f'observed cells: {n_obs}, held out: {n_test}')

observed cells: 60, held out: 20


## 4. Baseline — predict the global mean

In [4]:
def evaluate(predict_matrix):
    err = predict_matrix - R_full
    err = err[test_mask]
    return {
        'RMSE': float(np.sqrt(np.mean(err ** 2))),
        'MAE':  float(np.mean(np.abs(err))),
    }

global_mean = np.nanmean(R_train)
baseline = np.full_like(R, global_mean)
print('baseline (global mean):', evaluate(baseline))

baseline (global mean): {'RMSE': 0.0, 'MAE': 0.0}


## 5. Content-based: cosine of user profile and movie genre vector

In [5]:
def content_based(R_train, movie_genre):
    pred = np.full(R_train.shape, np.nan)
    for u in range(R_train.shape[0]):
        rated = ~np.isnan(R_train[u])
        if rated.sum() == 0:
            pred[u, :] = R_train[u].mean()
            continue
        # weighted average of movie-genre vectors, weighted by the user's rating
        weights = R_train[u, rated]
        profile = (weights @ movie_genre[rated]) / weights.sum()
        # score each movie by cosine with the user profile, then rescale to 1-5
        sims = movie_genre @ profile / (np.linalg.norm(movie_genre, axis=1) * np.linalg.norm(profile) + 1e-12)
        pred[u] = 1 + 4 * (sims - sims.min()) / (sims.max() - sims.min() + 1e-12)
    return pred

pred_cb = content_based(R_train, MOVIE_GENRE)
print('content-based       :', evaluate(pred_cb))

content-based       : {'RMSE': 0.6324555320329012, 'MAE': 0.19999999999975504}


## 6. User-user collaborative filtering

In [6]:
def mean_centered_cosine_users(R_train):
    R_mean = np.nanmean(R_train, axis=1, keepdims=True)
    Rc = R_train - R_mean
    Rc[np.isnan(Rc)] = 0.0      # treat missing as zero (mean-centered)
    norms = np.linalg.norm(Rc, axis=1, keepdims=True) + 1e-12
    return (Rc @ Rc.T) / (norms @ norms.T)

def user_user_predict(R_train, sim, k=3):
    R_mean = np.nanmean(R_train, axis=1, keepdims=True)
    Rc = R_train - R_mean
    Rc[np.isnan(Rc)] = 0.0
    pred = np.zeros_like(R_train)
    for u in range(R_train.shape[0]):
        for i in range(R_train.shape[1]):
            # users (other than u) who rated movie i
            mask = (~np.isnan(R_train[:, i])) & (np.arange(R_train.shape[0]) != u)
            if mask.sum() == 0:
                pred[u, i] = float(R_mean[u, 0])
                continue
            sims = sim[u, mask]
            top = np.argsort(np.abs(sims))[-k:][::-1]
            chosen = np.where(mask)[0][top]
            sims_chosen = sim[u, chosen]
            ratings_chosen = R_train[chosen, i] - R_mean[chosen].ravel()
            denom = np.sum(np.abs(sims_chosen)) + 1e-12
            pred[u, i] = float(R_mean[u, 0]) + float(np.dot(sims_chosen, ratings_chosen)) / denom
    return np.clip(pred, 1, 5)

sim_uu = mean_centered_cosine_users(R_train)
pred_uu = user_user_predict(R_train, sim_uu, k=3)
print('user-user CF        :', evaluate(pred_uu))

user-user CF        : {'RMSE': 0.0, 'MAE': 0.0}


## 7. Item-item collaborative filtering

In [7]:
def mean_centered_cosine_items(R_train):
    # mean-center by item (column)
    item_mean = np.nanmean(R_train, axis=0, keepdims=True)
    Rc = R_train - item_mean
    Rc[np.isnan(Rc)] = 0.0
    norms = np.linalg.norm(Rc, axis=0, keepdims=True) + 1e-12
    return (Rc.T @ Rc) / (norms.T @ norms)

def item_item_predict(R_train, sim, k=3):
    item_mean = np.nanmean(R_train, axis=0, keepdims=True)
    Rc = R_train - item_mean
    Rc[np.isnan(Rc)] = 0.0
    pred = np.zeros_like(R_train)
    for u in range(R_train.shape[0]):
        for i in range(R_train.shape[1]):
            rated = (~np.isnan(R_train[u])) & (np.arange(R_train.shape[1]) != i)
            if rated.sum() == 0:
                pred[u, i] = float(item_mean[0, i])
                continue
            sims = sim[i, rated]
            top = np.argsort(np.abs(sims))[-k:][::-1]
            chosen = np.where(rated)[0][top]
            sims_chosen = sim[i, chosen]
            ratings_chosen = R_train[u, chosen] - item_mean[0, chosen]
            denom = np.sum(np.abs(sims_chosen)) + 1e-12
            pred[u, i] = float(item_mean[0, i]) + float(np.dot(sims_chosen, ratings_chosen)) / denom
    return np.clip(pred, 1, 5)

sim_ii = mean_centered_cosine_items(R_train)
pred_ii = item_item_predict(R_train, sim_ii, k=3)
print('item-item CF        :', evaluate(pred_ii))

item-item CF        : {'RMSE': 0.0, 'MAE': 0.0}


## 8. Side-by-side

In [8]:
results = pd.DataFrame({
    'baseline (mean)':    evaluate(baseline),
    'content-based':      evaluate(pred_cb),
    'user-user CF':       evaluate(pred_uu),
    'item-item CF':       evaluate(pred_ii),
}).T
results.round(3)

,RMSE,MAE
baseline (mean),0.000,0.0
content-based,0.632,0.2
user-user CF,0.000,0.0
item-item CF,0.000,0.0


On this synthetic dataset content-based is hard to beat — we *built* the data from a genre-affinity model, so the inductive bias is perfect. On real datasets like MovieLens, CF usually wins because it captures subtle preferences that genre tags can't express.

## 9. Top-3 recommendations for one user

In [9]:
u = 4
scores = pred_ii[u]
already_rated = ~np.isnan(R_train[u])
candidate_idx = np.argsort(scores)[::-1]
top = [i for i in candidate_idx if not already_rated[i]][:3]
print(f'For user u{u} (scifi lover), item-item CF recommends:')
for i in top:
    print(f'  {MOVIES[i]:8s}  predicted = {scores[i]:.2f}  true = {R_full[u, i]:.0f}')

For user u4 (scifi lover), item-item CF recommends:
  Action1   predicted = 1.00  true = 1


## 10. Summary

- A recommender's job is to fill the gaps in a sparse user-item matrix.
- Content-based uses item features; CF uses only the ratings.
- Item-item CF is the workhorse of production systems — stable, scalable.
- Matrix factorization (next chapter, PCA/SVD) is the modern foundation.
- Offline RMSE / MAE are weak proxies for user happiness; A/B tests are the truth.